# Generate LLM Embeddings using GPT-4.0-mini via OpenAI API with .env support

In [1]:
# from openai import OpenAI
# import os
# from dotenv import load_dotenv
# import pandas as pd

# # Load API key from .env file
# load_dotenv()
# client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# def get_embedding(prompt, model="gpt-4o-mini"):
#     response = client.chat.completions.create(
#         model=model,
#         messages=[{"role": "user", "content": prompt}],
#         temperature=0.2,
#     )
#     return response.choices[0].message.content

# # Example usage with dummy transaction
# transaction_text = """
# Transaction of $450 made using Visa debit card at 9:45 AM on 12 July 2020, sent to vendor A1 Electronics.
# """
# embedding = get_embedding(transaction_text)
# print("LLM Output:", embedding)


In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm

# Load API key
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


In [3]:
# # Load training data
# df = pd.read_csv("../data/processed/train_full.csv")
   
# # Use small sample for demo/testing
# df = df.head(1000)  # You can increase this as needed

# # Fields to use in the prompt
# FIELDS = ['TransactionID', 'ProductCD', 'card1', 'addr1', 'DeviceType', 'DeviceInfo', 'R_emaildomain', 'P_emaildomain', 'isFraud']
# df = df[[col for col in FIELDS if col in df.columns]]


In [4]:
# # Create prompt from row
# def create_prompt(row):
#     prompt = f"A transaction was made using product {row.get('ProductCD', 'NA')} "
#     prompt += f"with card {row.get('card1', 'NA')}, from address {row.get('addr1', 'NA')}, "
#     prompt += f"on device {row.get('DeviceType', 'NA')} ({row.get('DeviceInfo', 'NA')}). "
#     prompt += f"The sender’s email domain was {row.get('P_emaildomain', 'NA')} and receiver’s email was {row.get('R_emaildomain', 'NA')}."
#     return prompt


In [5]:
# # Generate GPT-4o outputs
# results = []

# for _, row in tqdm(df.iterrows(), total=len(df)):
#     prompt = create_prompt(row)
#     try:
#         response = client.chat.completions.create(
#             model="gpt-4o-mini",
#             messages=[{"role": "user", "content": prompt}],
#             temperature=0.2
#         )
#         output = response.choices[0].message.content
#     except Exception as e:
#         output = f"ERROR: {str(e)}"
    
#     results.append({
#         "TransactionID": row['TransactionID'],
#         "Prompt": prompt,
#         "GPT4o_Output": output,
#         "isFraud": row.get("isFraud", -1)
#     })


In [6]:
# # Save output
# output_df = pd.DataFrame(results)
# os.makedirs("../data/processed", exist_ok=True)
# output_df.to_csv("../data/processed/gpt4o_outputs.csv", index=False)
# print("Saved to ../data/processed/gpt4o_outputs.csv")


In [7]:
# ... (load client and data as before)
df = pd.read_csv("../data/processed/train_full.csv").head(1000)

In [8]:
# Create an analytical prompt
def create_analytical_prompt(row):
    prompt = f"Analyze the following transaction: Product {row.get('ProductCD', 'NA')}, "
    prompt += f"card {row.get('card1', 'NA')}, address {row.get('addr1', 'NA')}, "
    prompt += f"device {row.get('DeviceType', 'NA')} ({row.get('DeviceInfo', 'NA')}), "
    prompt += f"sender's email {row.get('P_emaildomain', 'NA')}. "
    prompt += "What are its key characteristics?"
    return prompt

In [9]:
# Generate embeddings (not text)
results = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    prompt = create_analytical_prompt(row)
    try:
        # FIXED: Use the embeddings API, not the chat completions API
        response = client.embeddings.create(
            model="text-embedding-3-small",  # This is a good, low-cost embedding model
            input=prompt
        )
        embedding = response.data[0].embedding
    except Exception as e:
        embedding = [0.0] * 1536 # Default to zeros on error
    
    results.append({
        "TransactionID": row['TransactionID'],
        "Prompt": prompt,
        "LLM_Embedding": embedding
    })

100%|██████████| 1000/1000 [07:32<00:00,  2.21it/s]


In [1]:
 # Save the embeddings
output_df = pd.DataFrame(results)
# Flatten the list of embeddings into a single dataframe
output_df_flat = pd.json_normalize(output_df['LLM_Embedding']).add_prefix('LLM_embed_')
output_df = pd.concat([output_df.drop('LLM_Embedding', axis=1), output_df_flat], axis=1)

output_df.to_csv("../data/processed/llm_embeddings.csv", index=False)

NameError: name 'pd' is not defined